In [1]:


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

fact_sales_fe=pd.read_csv(r"C:\Users\RohanS\Documents\retail project\data retail processed\fact_sales_fe.csv")
df = fact_sales_fe.copy()



df["order_delivered_customer_date"] = pd.to_datetime(
    df["order_delivered_customer_date"]
)

df["order_estimated_delivery_date"] = pd.to_datetime(
    df["order_estimated_delivery_date"]
)


df["delivery_delay"] = (
    df["order_delivered_customer_date"]
    >
    df["order_estimated_delivery_date"]
).astype(int)



features = [
    "price",
    "freight_value",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

model_df = df[
    features + ["delivery_delay"]
].copy()




model_df = model_df.dropna()




X = model_df[features]

y = model_df["delivery_delay"]


print("Dataset shape:", model_df.shape)

print("\nDelay distribution:")
print(
    y.value_counts()
)



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(
    X_train,
    y_train
)



y_pred = model.predict(
    X_test
)




print("\nAccuracy:")
print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


importance = pd.DataFrame({

    "feature": features,

    "importance": model.feature_importances_

}).sort_values(
    "importance",
    ascending=False
)


print("\nFeature Importance:")
print(importance)



df_model = df[
    features
].copy()

df_model = df_model.fillna(
    df_model.median(numeric_only=True)
)


df["delay_probability"] = (
    model.predict_proba(
        df_model
    )[:, 1]
)



df["delay_risk"] = pd.cut(

    df["delay_probability"],

    bins=[
        -np.inf,
        0.30,
        0.60,
        np.inf
    ],

    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)


delay_risk = df[
    [
        "order_id",
        "product_id",
        "seller_id",
        "delay_probability",
        "delay_risk"
    ]
].copy()


delay_risk["delay_probability"] = (
    delay_risk["delay_probability"]
    .round(3)
)


print("\nDelay risk:")
print(
    delay_risk.head(20)
)


print("\nRisk distribution:")
print(
    delay_risk["delay_risk"]
    .value_counts()
)



delay_risk.to_csv(
    "C:/Users/RohanS/Desktop/delivery_delay_predictions_8_2.csv",
    index=False
)

importance.to_csv(
    "C:/Users/RohanS/Desktop/delay_feature_importance_8_2.csv",
    index=False
)


Dataset shape: (118290, 7)

Delay distribution:
delivery_delay
0    109222
1      9068
Name: count, dtype: int64

Accuracy:
0.8907346352185307

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94     21844
           1       0.28      0.28      0.28      1814

    accuracy                           0.89     23658
   macro avg       0.61      0.61      0.61     23658
weighted avg       0.89      0.89      0.89     23658


Confusion Matrix:
[[20569  1275]
 [ 1310   504]]

Feature Importance:
             feature  importance
1      freight_value    0.276703
0              price    0.225365
2   product_weight_g    0.172275
4  product_height_cm    0.118312
3  product_length_cm    0.109515
5   product_width_cm    0.097830

Delay risk:
                            order_id                        product_id  \
0   e481f51cbdc54678b7cc49136f2d6af7  87285b34884572647811a353c7ac498a   
1   e481f51cbdc54678b7cc49136f2d6af7  87285